# PyTorX - VGG9 CIFAR-10 Benchmark

This notebook trains a VGG9 network on CIFAR-10 using PyTorX crossbar-aware layers.

**Non-ideal effects available:** IR drop, thermal/RTN noise, stuck-at-fault (SAF), DAC/ADC quantization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fouda-compumacy/pytorx/blob/alpha/notebooks/cifar10_vgg9.ipynb)

## 1. Setup

In [ ]:
import os

if not os.path.exists('pytorx'):
    !git clone https://github.com/fouda-compumacy/pytorx.git

os.chdir('pytorx')

!pip install -q torch torchvision numpy scipy pandas matplotlib

In [ ]:
import sys
import shutil
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from IPython.display import clear_output

from python.torx.module.layer import crxb_Conv2d, crxb_Linear

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Configuration

In [ ]:
class Config:
    # Training
    epochs = 200
    batch_size = 128
    test_batch_size = 100
    lr = 0.05
    momentum = 0.9
    weight_decay = 1e-4
    seed = 1

    # Crossbar
    crxb_size = 64
    vdd = 3.3
    gwire = 0.375
    gload = 0.25
    gmax = 0.000333
    gmin = 0.0000000333
    freq = 10e6
    temp = 300
    scaler_dw = 1
    quantize = 8
    weight_precision = None

    # Non-ideal effects
    ir_drop = False
    enable_noise = False
    enable_SAF = False
    enable_ec_SAF = False

args = Config()
torch.manual_seed(args.seed)

## 3. Data Loading

In [ ]:
kwargs = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10('./data', train=True, download=True, transform=transform_train),
    batch_size=args.batch_size, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10('./data', train=False, transform=transform_test),
    batch_size=args.test_batch_size, shuffle=False, **kwargs)

print(f'Train: {len(train_loader.dataset)} samples, Test: {len(test_loader.dataset)} samples')

## 4. Model Definition

In [ ]:
class VGG9(nn.Module):
    """VGG9: Conv128->Conv128->Pool -> Conv256->Conv256->Pool -> Conv512->Conv512->Pool -> FC1024->FC1024->FC10"""

    def __init__(self, cfg, num_classes=10):
        super(VGG9, self).__init__()
        crxb_cfg = dict(
            crxb_size=cfg.crxb_size, scaler_dw=cfg.scaler_dw,
            gwire=cfg.gwire, gload=cfg.gload, gmax=cfg.gmax, gmin=cfg.gmin,
            vdd=cfg.vdd, freq=cfg.freq, temp=cfg.temp,
            enable_SAF=cfg.enable_SAF, enable_ec_SAF=cfg.enable_ec_SAF,
            enable_noise=cfg.enable_noise, ir_drop=cfg.ir_drop,
            device=device, quantize=cfg.quantize,
            weight_precision=cfg.weight_precision)

        self.conv1 = crxb_Conv2d(3, 128, kernel_size=3, padding=1, **crxb_cfg)
        self.bn1 = nn.BatchNorm2d(128)
        self.conv2 = crxb_Conv2d(128, 128, kernel_size=3, padding=1, **crxb_cfg)
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = crxb_Conv2d(128, 256, kernel_size=3, padding=1, **crxb_cfg)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = crxb_Conv2d(256, 256, kernel_size=3, padding=1, **crxb_cfg)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = crxb_Conv2d(256, 512, kernel_size=3, padding=1, **crxb_cfg)
        self.bn5 = nn.BatchNorm2d(512)
        self.conv6 = crxb_Conv2d(512, 512, kernel_size=3, padding=1, **crxb_cfg)
        self.bn6 = nn.BatchNorm2d(512)

        self.fc1 = crxb_Linear(512 * 4 * 4, 1024, **crxb_cfg)
        self.bn7 = nn.BatchNorm1d(1024)
        self.fc2 = crxb_Linear(1024, 1024, **crxb_cfg)
        self.bn8 = nn.BatchNorm1d(1024)
        self.fc3 = crxb_Linear(1024, num_classes, **crxb_cfg)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(F.relu(self.bn4(self.conv4(x))), 2)
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.max_pool2d(F.relu(self.bn6(self.conv6(x))), 2)
        x = x.view(x.size(0), -1)
        x = F.dropout(F.relu(self.bn7(self.fc1(x))), 0.5, training=self.training)
        x = F.dropout(F.relu(self.bn8(self.fc2(x))), 0.5, training=self.training)
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)


model = VGG9(args, num_classes=10).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=args.lr,
                      momentum=args.momentum, weight_decay=args.weight_decay)
criterion = nn.NLLLoss().to(device)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0.0, 0
    for data, target in loader:
        for m in model.modules():
            if isinstance(m, (crxb_Conv2d, crxb_Linear)):
                m._reset_delta()
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += output.argmax(1).eq(target).sum().item()
    n = len(loader.dataset)
    return total_loss / n, 100. * correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item() * data.size(0)
            correct += output.argmax(1).eq(target).sum().item()
    n = len(loader.dataset)
    return total_loss / n, 100. * correct / n

In [ ]:
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(1, args.epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, test_loader, criterion)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_accs.append(tr_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_vgg9_cifar10.pth')

    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_losses, label='Train')
    ax1.plot(val_losses, label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
    ax2.plot(train_accs, label='Train')
    ax2.plot(val_accs, label='Val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.legend(); ax2.set_title('Accuracy')
    plt.tight_layout()
    plt.show()

    print(f'Epoch {epoch}/{args.epochs}  LR={optimizer.param_groups[0]["lr"]:.6f}  '
          f'Train Loss={tr_loss:.4f}  Train Acc={tr_acc:.2f}%  '
          f'Val Loss={val_loss:.4f}  Val Acc={val_acc:.2f}%  '
          f'Best={best_val_acc:.2f}%')

## 6. Save Checkpoint

In [ ]:
torch.save({
    'epoch': epoch,
    'state_dict': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'val_acc': best_val_acc,
}, f'checkpoint_cifar10_{args.crxb_size}.pth.tar')
print(f'Saved. Best val accuracy: {best_val_acc:.2f}%')